In [5]:
pip install -q psycopg2-binary tqdm

Python(20451) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.



[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import json
import re
import psycopg2
import psycopg2.extras
from tqdm.auto import tqdm

In [ ]:
# ==========================================
# CẤU HÌNH
# ==========================================

DB_HOST     = "smartphone-db.cbmw80kis21l.ap-southeast-2.rds.amazonaws.com"
DB_NAME     = "postgres"
DB_USER     = "postgres"
DB_PASSWORD = "admindata123"

FETCH_SIZE   = 5000
COMMIT_EVERY = 1000

KEYWORDS = {
    "camera": [
        # VI
        "camera", "chụp", "ảnh", "zoom", "selfie", "quay", "phim",
        "góc rộng", "telephoto", "chụp đêm", "chụp ngày", "xóa phông",
        "chân dung", "ống kính", "siêu rộng", "macro",
        "cam", "chụp hình", "camera trước", "camera sau",
        "cam trước", "cam sau", "quay video", "lấy nét", "thiếu sáng",
        # EN
        "photo", "picture", "shoot", "lens", "portrait", "video",
        "footage", "nightmode", "bokeh", "ultrawide", "snapshot",
    ],
    "battery": [
        # VI
        "pin", "sạc", "hao pin", "nạp điện", "sạc nhanh", "sạc không dây",
        "cạn pin", "trâu pin", "pin trâu", "thời lượng pin",
        "trâu", "tụt", "tuột", "hao", "bin",
        "sạc pin", "hết pin", "pin tụt", "pin khỏe",
        "qua đêm", "nhanh hết", "tụt nhanh", "pin tuột", "mau hết",
        # EN
        "battery", "charge", "charging", "drain", "fast charge",
        "wireless charge", "battery life", "power bank",
    ],
    "display": [
        # VI
        "màn hình", "màn", "độ sáng", "tần số quét", "ám vàng",
        "burn-in", "tấm nền", "độ phân giải", "notch", "đục lỗ",
        "oled", "amoled", "lcd", "hiển thị", "phân giải", "rực rỡ",
        # EN
        "screen", "display", "nit", "refresh rate", "resolution",
        "panel", "brightness", "pwm", "punchhole",
    ],
    "performance": [
        # VI
        "hiệu năng", "lag", "mượt", "chậm", "giật", "nóng máy",
        "đơ", "nhanh", "hiệu suất", "tản nhiệt", "chip",
        "vi xử lý", "xử lý", "ram",
        "game", "chơi game", "cấu hình", "chơi", "liên quân", "pubg",
        "giật lag", "ứng dụng", "đa nhiệm", "phần mềm", "khởi động", "lướt web",
        # EN
        "snapdragon", "dimensity", "cpu", "gpu", "processor",
        "benchmark", "smooth", "heating", "throttle", "performance",
        "helio", "gaming", "freeze",
    ],
    "design": [
        # VI
        "thiết kế", "mỏng", "nhẹ", "nặng", "màu sắc", "vỏ máy",
        "kính", "nhôm", "titan", "nhựa", "sang trọng", "ngoại hình",
        "mặt lưng", "vừa tay", "mẫu mã", "kiểu dáng", "nhỏ gọn",
        "mỏng nhẹ", "bắt mắt", "cầm chắc", "lưng nhựa", "thời trang",
        # EN
        "design", "build quality", "color", "weight", "thin", "premium",
        "glass", "aluminum", "titanium", "finish", "aesthetic",
    ],
    "storage": [
        # VI
        "bộ nhớ", "lưu trữ", "dung lượng", "rom", "thẻ nhớ",
        "bộ nhớ trong", "bộ nhớ ngoài", "đầy bộ nhớ",
        # EN
        "storage", "sd card", "microsd", "internal storage", "capacity",
    ],
    "connectivity": [
        # VI
        "wifi", "sóng", "kết nối", "sim", "mạng", "hotspot", "jack tai nghe",
        "bắt sóng", "bắt wifi", "sóng wifi",
        # EN
        "bluetooth", "nfc", "network", "signal", "lte", "type-c",
        "5g", "4g", "3g",
    ],
    "utilities": [
        # VI
        "tính năng", "vân tay", "chống nước", "nhận diện khuôn mặt",
        "loa ngoài", "âm thanh", "stereo", "always on display",
        "cảm ứng", "nhận diện", "mở khóa", "nghe nhạc", "tai nghe",
        "cảm biến vân tay",
        # EN
        "face id", "fingerprint", "ip68", "ip67", "waterproof",
        "speaker", "audio", "dolby", "haptic", "aod",
    ],
}

# Unit patterns — số + đơn vị kỹ thuật: "120Hz", "64MP", "5000mAh", "256GB"
UNIT_PATTERNS = {
    "display":  [r"\d+\s*hz"],
    "camera":   [r"\d+\s*mp"],
    "battery":  [r"\d+\s*mah", r"\d+\s*w(?:att)?(?:\s|$)"],
    "storage":  [r"\d+\s*(?:gb|tb)"],
}

ASPECT_TO_TABLE = {
    "display":      "dim_display",
    "camera":       "dim_camera",
    "battery":      "dim_battery",
    "performance":  "dim_performance",
    "design":       "dim_design",
    "storage":      "dim_storage",
    "connectivity": "dim_connectivity",
    "utilities":    "dim_utilities",
}

In [9]:
# ==========================================
# KẾT NỐI DB
# ==========================================

conn = psycopg2.connect(
    host=DB_HOST, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD
)
conn.autocommit = False
cursor = conn.cursor()
print("✅ Kết nối DB thành công.")

✅ Kết nối DB thành công.


In [10]:
# ==========================================
# HÀM KEYWORD MATCH
# ==========================================
# Precompile toàn bộ pattern 1 lần duy nhất để tăng tốc

_KW_COMPILED = {
    aspect: [
        re.compile(r"\b" + re.escape(kw) + r"\b", re.IGNORECASE)
        for kw in kws
    ]
    for aspect, kws in KEYWORDS.items()
}

_UNIT_COMPILED = {
    aspect: [re.compile(pat, re.IGNORECASE) for pat in pats]
    for aspect, pats in UNIT_PATTERNS.items()
}


def keyword_match(text: str) -> dict:
    matched = {}

    for aspect, patterns in _KW_COMPILED.items():
        for pat in patterns:
            if pat.search(text):
                matched[aspect] = 1
                break

    for aspect, patterns in _UNIT_COMPILED.items():
        if aspect not in matched:
            for pat in patterns:
                if pat.search(text):
                    matched[aspect] = 1
                    break

    return matched


# Smoke test
samples = [
    ("camera chụp đêm rất nét",              {"camera"}),
    ("pin trâu dùng được 2 ngày",            {"battery"}),
    ("màn hình 120Hz mượt lắm",              {"display", "performance"}),
    ("256GB lưu trữ thoải mái",              {"storage"}),
    ("Wo sab to theek h lekin manoj bhai",   set()),
    ("@AnLuongVinh bạn pass không?",         set()),
    ("64MP chụp đẹp",                        {"camera"}),
    ("5000mAh sạc nhanh 65W",               {"battery"}),
]
all_ok = True
for text, expected in samples:
    result = set(keyword_match(text).keys())
    ok = expected.issubset(result)
    status = "✅" if ok else "❌"
    print(f"{status} {text!r:45s} → {result}")
    if not ok:
        all_ok = False
print("\n✅ Smoke test passed" if all_ok else "\n❌ Có case chưa đúng")

✅ 'camera chụp đêm rất nét'                     → {'camera'}
✅ 'pin trâu dùng được 2 ngày'                   → {'battery'}
✅ 'màn hình 120Hz mượt lắm'                     → {'display', 'performance'}
✅ '256GB lưu trữ thoải mái'                     → {'storage'}
✅ 'Wo sab to theek h lekin manoj bhai'          → set()
✅ '@AnLuongVinh bạn pass không?'                → set()
✅ '64MP chụp đẹp'                               → {'camera'}
✅ '5000mAh sạc nhanh 65W'                       → {'performance', 'battery'}

✅ Smoke test passed


In [11]:
# ==========================================
# BƯỚC 1: CLASSIFY TỪNG COMMENT (multi-label)
# ==========================================
# Resume tự động: chỉ xử lý comment có aspects IS NULL

try:
    conn.close()
except Exception:
    pass

conn = psycopg2.connect(
    host=DB_HOST, dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD,
    connect_timeout=10,
    options="-c statement_timeout=120000",
)
conn.autocommit = False

# withhold=True → DECLARE ... WITH HOLD → cursor tồn tại sau mỗi conn.commit()
read_cursor  = conn.cursor("classify_read", withhold=True)
write_cursor = conn.cursor()

print("Đang fetch comments chưa classify...")
read_cursor.execute("""
    SELECT comment_id, comment_text
    FROM dim_video_comments
    WHERE aspects IS NULL
      AND LENGTH(TRIM(comment_text)) >= 3
""")
print("Query OK — bắt đầu classify.")

processed     = 0
fetch_num     = 0
update_buffer = []

while True:
    rows = read_cursor.fetchmany(FETCH_SIZE)
    if not rows:
        break

    fetch_num += 1
    matched_count = 0

    for comment_id, comment_text in rows:
        aspects_dict = keyword_match(comment_text)
        if aspects_dict:
            matched_count += 1
        update_buffer.append((json.dumps(aspects_dict, ensure_ascii=False), comment_id))

    print(f"[Fetch {fetch_num}] {len(rows):,} comments | match: {matched_count:,} | Đã xử lý: {processed:,}")

    if len(update_buffer) >= COMMIT_EVERY:
        psycopg2.extras.execute_batch(
            write_cursor,
            "UPDATE dim_video_comments SET aspects = %s::jsonb WHERE comment_id = %s",
            update_buffer,
        )
        conn.commit()
        processed    += len(update_buffer)
        update_buffer = []
        print(f"  ✔ Commit | {processed:,}")

# flush cuối
if update_buffer:
    psycopg2.extras.execute_batch(
        write_cursor,
        "UPDATE dim_video_comments SET aspects = %s::jsonb WHERE comment_id = %s",
        update_buffer,
    )
    conn.commit()
    processed += len(update_buffer)

read_cursor.close()
write_cursor.close()
print(f"\n✅ Bước 1 hoàn tất. Tổng: {processed:,} comment")

Đang fetch comments chưa classify...
Query OK — bắt đầu classify.
[Fetch 1] 5,000 comments | match: 1,553 | Đã xử lý: 0
  ✔ Commit | 5,000
[Fetch 2] 5,000 comments | match: 1,362 | Đã xử lý: 5,000
  ✔ Commit | 10,000
[Fetch 3] 5,000 comments | match: 1,604 | Đã xử lý: 10,000
  ✔ Commit | 15,000
[Fetch 4] 5,000 comments | match: 1,497 | Đã xử lý: 15,000
  ✔ Commit | 20,000
[Fetch 5] 5,000 comments | match: 1,494 | Đã xử lý: 20,000
  ✔ Commit | 25,000
[Fetch 6] 5,000 comments | match: 1,549 | Đã xử lý: 25,000
  ✔ Commit | 30,000
[Fetch 7] 5,000 comments | match: 1,052 | Đã xử lý: 30,000
  ✔ Commit | 35,000
[Fetch 8] 5,000 comments | match: 1,525 | Đã xử lý: 35,000
  ✔ Commit | 40,000
[Fetch 9] 5,000 comments | match: 1,304 | Đã xử lý: 40,000
  ✔ Commit | 45,000
[Fetch 10] 5,000 comments | match: 1,699 | Đã xử lý: 45,000
  ✔ Commit | 50,000
[Fetch 11] 5,000 comments | match: 1,412 | Đã xử lý: 50,000
  ✔ Commit | 55,000
[Fetch 12] 5,000 comments | match: 945 | Đã xử lý: 55,000
  ✔ Commit |

KeyboardInterrupt: 

In [ ]:
# ==========================================
# KIỂM TRA KẾT QUẢ BƯỚC 1
# ==========================================

cursor = conn.cursor()

cursor.execute("""
    SELECT aspect_key, COUNT(*) AS so_luong
    FROM dim_video_comments,
         jsonb_object_keys(aspects) AS aspect_key
    WHERE aspects IS NOT NULL AND aspects != '{}'
    GROUP BY aspect_key
    ORDER BY so_luong DESC
""")

print(f"{'Aspect':<15} {'Số comment':>12}")
print("-" * 28)
for row in cursor.fetchall():
    print(f"{row[0]:<15} {row[1]:>12,}")

cursor.execute("SELECT COUNT(*) FROM dim_video_comments WHERE aspects = '{}'")
irrelevant = cursor.fetchone()[0]
cursor.execute("SELECT COUNT(*) FROM dim_video_comments WHERE aspects IS NULL")
not_yet = cursor.fetchone()[0]
print(f"\nKhông liên quan (aspects={{}}): {irrelevant:,}")
print(f"Chưa classify (aspects IS NULL): {not_yet:,}")

In [ ]:
# ==========================================
# BƯỚC 2: TỔNG HỢP VÀO user_review CỦA 8 BẢNG DIM
# ==========================================
# Một comment có aspects={"camera": 1, "display": 1}
# sẽ được đưa vào CẢ HAI dim_camera.user_review và dim_display.user_review

cursor = conn.cursor()

for aspect_en, dim_table in tqdm(ASPECT_TO_TABLE.items(), desc="Ghi user_review"):
    cursor.execute(f"""
        SELECT
            v.product_id,
            json_agg(c.comment_text ORDER BY c.comment_id) AS comments,
            COUNT(*) AS total
        FROM dim_video_comments c
        JOIN dim_video_transcripts v ON c.video_id = v.video_id
        WHERE c.aspects ? %s
        GROUP BY v.product_id
    """, (aspect_en,))

    rows = cursor.fetchall()

    for product_id, comments_list, total in rows:
        user_review_json = json.dumps(
            {"total": total, "comments": comments_list},
            ensure_ascii=False
        )
        cursor.execute(f"""
            UPDATE {dim_table}
            SET user_review  = %s,
                last_updated = NOW()
            WHERE product_id = %s
        """, (user_review_json, product_id))

    conn.commit()
    print(f"✅ {dim_table}: đã cập nhật {len(rows):,} sản phẩm")

In [ ]:
# ==========================================
# KIỂM TRA KẾT QUẢ BƯỚC 2
# ==========================================

cursor.execute("""
    SELECT 'dim_camera'       AS bang, COUNT(*) FROM dim_camera       WHERE user_review IS NOT NULL
    UNION ALL
    SELECT 'dim_display',              COUNT(*) FROM dim_display       WHERE user_review IS NOT NULL
    UNION ALL
    SELECT 'dim_battery',              COUNT(*) FROM dim_battery       WHERE user_review IS NOT NULL
    UNION ALL
    SELECT 'dim_performance',          COUNT(*) FROM dim_performance   WHERE user_review IS NOT NULL
    UNION ALL
    SELECT 'dim_design',               COUNT(*) FROM dim_design        WHERE user_review IS NOT NULL
    UNION ALL
    SELECT 'dim_storage',              COUNT(*) FROM dim_storage       WHERE user_review IS NOT NULL
    UNION ALL
    SELECT 'dim_connectivity',         COUNT(*) FROM dim_connectivity  WHERE user_review IS NOT NULL
    UNION ALL
    SELECT 'dim_utilities',            COUNT(*) FROM dim_utilities     WHERE user_review IS NOT NULL
    ORDER BY 2 DESC
""")

print(f"{'Bảng':<20} {'Số sản phẩm có user_review':>26}")
print("-" * 47)
for row in cursor.fetchall():
    print(f"{row[0]:<20} {row[1]:>26,}")

cursor.close()
conn.close()
print("Đã đóng kết nối. Pipeline hoàn tất.")